In [1]:
from typing import TypedDict, Optional, List, Dict
from langgraph.graph import StateGraph, END


class MASState(TypedDict):
    user_input: str
    intent: Optional[dict]
    clarification_needed: Optional[bool]
    clarification_question: Optional[str]
    plan: Optional[dict]
    summary: Optional[str]
    evaluation: Optional[dict]
    final_output: Optional[str]

In [2]:
import uuid
from pathlib import Path
import sys
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Source' / 'ai').exists()), None)
if project_root and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from Source.ai.Multi_Agent_System.Agents.Orchestrator import Orchestrator
from Source.ai.Multi_Agent_System.Agents.Abstracter_Agent import AbstracterAgent
from Source.ai.Multi_Agent_System.Agents.Extractor_Agent import ExtractorAgent
from Source.ai.Multi_Agent_System.Memory.memory import MemoryManager
from Source.ai.Multi_Agent_System.Memory.experience_memory import ExperienceMemory
from Source.ai.Multi_Agent_System.Agents.Intent_Agent import IntentAgent
from Source.ai.Multi_Agent_System.Agents.Coordinator_Agent import CoordinatorAgent
from Source.ai.Multi_Agent_System.Agents.SessionMemory import SessionMemory
from Source.ai.Multi_Agent_System.Agents.ConversationManager import ConversationManager
from Source.ai.Multi_Agent_System.Agents.Clarification_Agent import ClarificationAgent
from Source.ai.Multi_Agent_System.Agents.Planning_Agent import PlanningAgent
from Source.ai.Multi_Agent_System.Agents.Evaluation_Agent import EvaluationAgent

# Initialize agents
intent_agent = IntentAgent()
clarification_agent = ClarificationAgent()
planning_agent = PlanningAgent()
evaluation_agent = EvaluationAgent()

abstracter = AbstracterAgent(
    model_path="../../../Model Train/Model_DG/vit5_grade_summary"
)

extractor = ExtractorAgent(
    model_path=r"E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_TX\vubert_summary_model.pth"
)

system1_engine = Orchestrator(
    abstracter_agent=abstracter,
    extractor_agent=extractor
)

print("MAS Agents initialized successfully.")

e:\Project_NguyenMinhVu_2211110063\Source\ai\Multi_Agent_System\Agents\Intent_Agent.py:23: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  self.llm = ChatOllama(model="qwen2.5:7b")
c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
e:\Project_NguyenMinhVu_2211110063\Source\ai\Multi_Agent_System\Agents\Extractor_Agent.py:78: FutureWarning

MAS Agents initialized successfully.


In [3]:
def intent_node(state: MASState):
    intent_result = intent_agent.classify(state["user_input"])
    return {"intent": intent_result}

def clarification_node(state: MASState):
    clarification = clarification_agent.analyze(
        state["user_input"],
        state["intent"]
    )

    if clarification["need_clarification"]:
        return {
            "clarification_needed": True,
            "clarification_question": clarification["question"],
            "final_output": clarification["question"]
        }

    return {"clarification_needed": False}

def planning_node(state: MASState):
    plan = planning_agent.plan(state["intent"])
    return {"plan": plan}

def summarize_node(state: MASState):
    
    pipeline = state["plan"]["pipeline"]

    for step in pipeline:
        if step["step"] == "summarize":
            summary = system1_engine.run(
                user_input=state["user_input"],
                strategy=step.get("strategy"),
                grade_level=step.get("grade_level")
            )
            return {"summary": summary}

    return {}

def evaluation_node(state: MASState):
    
    pipeline = state["plan"]["pipeline"]

    for step in pipeline:
        if step["step"] == "evaluate":
            eval_result = evaluation_agent.evaluate(
                text=state["user_input"],
                summary=state["summary"],
                metrics=step.get("metrics", [])
            )

            return {
                "evaluation": eval_result,
                "final_output": f"{state['summary']}\n\nĐánh giá:\n{eval_result}"
            }

    return {
        "final_output": state["summary"]
    }

In [4]:
builder = StateGraph(MASState)

builder.add_node("intent", intent_node)
builder.add_node("clarification", clarification_node)
builder.add_node("planning", planning_node)
builder.add_node("summarize", summarize_node)
builder.add_node("evaluate", evaluation_node)

builder.set_entry_point("intent")

builder.add_edge("intent", "clarification")

# Conditional routing
def route_after_clarification(state: MASState):
    if state.get("clarification_needed"):
        return END
    return "planning"

builder.add_conditional_edges(
    "clarification",
    route_after_clarification
)

builder.add_edge("planning", "summarize")
builder.add_edge("summarize", "evaluate")
builder.add_edge("evaluate", END)

graph = builder.compile()

In [5]:
def run_mas(user_input: str):
    initial_state = {
        "user_input": user_input
    }

    result = graph.invoke(initial_state)
    return result.get("final_output")

In [7]:
response = run_mas(
    """Hãy tóm tắt diễn giải theo lớp 1 đoạn văn sau: Níc Vu-gic (Nick Vujicic) là một diễn 
giả truyền cảm hứng người Úc, sinh năm 
1982. Khi chào đời, anh không có tứ chi 
mà chỉ có một bàn chân với hai ngón 
chân nhỏ. Sự khác biệt về ngoại hình 
khiến Níc từng bị bạn bè chê cười, trêu 
chọc. Đó là thời kì vô cùng khó khăn, 
Níc Vu-gic đã chấp nhận chung sống 
với sự thiếu sót trên cơ thể mình. Anh học cách dùng chân và một cái cán để 
viết chữ, đánh bàn phím máy vi tính, tự sinh hoạt cá nhân, chăm sóc bản thân. 
Anh đã vượt qua khó khăn, biến bản thân mình thành điều kì diệu trong cuộc 
sống. Níc Vu-gic trở thành biểu tượng mãnh liệt của nghị lực sống và lan toả 
năng lượng tích cực tới người xung quanh."""
)

print(response)

c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Níc Vu-gic là một diễn giả truyền cảm hứng người Úc, sinh năm 1982 và chỉ có một bàn chân với hai ngón chân nhỏ. Sự khác biệt về ngoại hình khiến Níc từng bị bạn bè chê cười và trêu chọc. Trong thời gian khó khăn, Níc đã học cách dùng chân và một cái cán để viết chữ, đánh bàn phím máy vi tính và chăm sóc bản thân. Anh đã vượt qua khó khăn và biến bản thân thành điều kì diệu trong cuộc sống.

Đánh giá:
{'rouge1_precision': 1.0, 'rouge1_recall': 0.5535248041775457, 'rouge1_f1': 0.7126050420168066, 'rougeL_precision': 0.9481132075471698, 'rougeL_recall': 0.5248041775456919, 'rougeL_f1': 0.6756302521008403, 'bertscore_precision': 0.9016571640968323, 'bertscore_recall': 0.8673199415206909, 'bertscore_f1': 0.8841552138328552, 'comment': 'Tóm tắt có mức độ bao phủ nội dung khá tốt. Mức độ tương đồng ngữ nghĩa rất cao.'}
